# Autoencoder implementation

This notebook implements autoencoder using 1 year hourly T_2M data 

In [70]:
from pathlib import Path
import sys
import importlib
import numpy as np
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / 'src' / 'my_ml_zoo').exists():
        repo_root = candidate
        break
else:
    raise RuntimeError('Unable to locate repository root containing src/my_ml_zoo')

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

importlib.invalidate_caches()
for module_name in [
    'my_ml_zoo.data',
    'my_ml_zoo',
    'my_ml_zoo.data.dataset_config',
    'my_ml_zoo.data.file_discovery',
    'my_ml_zoo.data.xarray_dataset',
]:
    if module_name in sys.modules:
        del sys.modules[module_name]

from my_ml_zoo.data import (
    DatasetConfig,
    discover_variable_files,
    select_variable_file,
    open_variable_dataset,
    infer_file_year,
)


### 1. Define dataset 'IAEVALL03'

In [6]:
config_path = repo_root / 'configs' / 'datasets' / 'IAEVALL03.json'
dataset_config = DatasetConfig.load_from_json(config_path)

### 2. Extract the T_2M data for the year 2002

In [7]:
ds_t2m_2000 = open_variable_dataset(dataset_config, 'T_2M', year=2000)['T_2M']
ds_t2m_2000 = ds_t2m_2000.squeeze('height')

n_time, nx, ny = ds_t2m_2000.shape
print(f'( n_time, nx, ny ) = ({n_time}, {nx}, {ny})')

( n_time, nx, ny ) = (8784, 412, 424)


### 3. Divide the data to:
1. training (70%)
2. validation (15%)
3. test (15%)

the split is none consequtively within each month (to get rid of seasonal bias)

In [63]:
rng = np.random.default_rng(42)

months = ds_t2m_2000["time"].dt.month.values

train_idx = []
val_idx = []
test_idx = []

for month in range(1, 13):
    
    idx_month = np.where(months == month)[0]
    
    idx_month = rng.permutation(idx_month)

    n_curr_month = len(idx_month)

    n_train_curr_month = int(0.70 * n_curr_month)
    n_val_curr_month = int(0.15 * n_curr_month)
    n_test_curr_month = n_curr_month - n_train_curr_month - n_val_curr_month

    train_idx.append(idx_month[:n_train_curr_month])
    val_idx.append(idx_month[n_train_curr_month:n_train_curr_month + n_val_curr_month])
    test_idx.append(idx_month[n_train_curr_month + n_val_curr_month:])

train_idx = np.sort(np.concatenate(train_idx))
val_idx = np.sort(np.concatenate(val_idx))
test_idx = np.sort(np.concatenate(test_idx))

train_da = ds_t2m_2000.isel(time=train_idx)
val_da   = ds_t2m_2000.isel(time=val_idx)
test_da  = ds_t2m_2000.isel(time=test_idx)

print('train da shape:',train_da.shape)
print('val da shape:',val_da.shape)
print('test da shape:',test_da.shape)

train da shape: (6139, 412, 424)
val da shape: (1313, 412, 424)
test da shape: (1332, 412, 424)


### 4. Compute the normalization characteristics on the training data

In [64]:
train_mean = train_da.astype(np.float64).mean().item()
train_std = train_da.astype(np.float64).std().item()

print("train_mean =", train_mean)
print("train_std  =", train_std)

train_mean = 285.0142155784381
train_std  = 10.2971076818789


### 5. Normalize the training, validation and test data using the obtained characteristics

In [65]:
train_norm = (train_da - train_mean) / train_std
val_norm   = (val_da   - train_mean) / train_std
test_norm  = (test_da  - train_mean) / train_std

In [66]:
# validation
print("train norm mean:", train_norm.astype(np.float64).mean().item())
print("train norm std :", train_norm.astype(np.float64).std().item())
print("val   norm mean:", val_norm.astype(np.float64).mean().item())
print("val   norm std :", val_norm.astype(np.float64).std().item())

train norm mean: -5.451027059425876e-07
train norm std : 0.9999999985746307
val   norm mean: 0.00295235601495781
val   norm std : 1.0027945030487742


### 6. Save the data as numpy array

In [72]:
X_train = train_norm.values.astype(np.float32)
X_val   = val_norm.values.astype(np.float32)
X_test  = test_norm.values.astype(np.float32)

## 7. Add the channel dimension 

PyTorch convolution layers expect image-like data in the shape [N, C, H, W]

1. N - batch size
2. C - number of channels
3. H - height
4. W - width

In [73]:
print("before channel dim:")
print("X_train:", X_train.shape, X_train.dtype)
print("X_val:", X_val.shape, X_val.dtype)
print("X_test:", X_test.shape, X_test.dtype)

X_train = X_train[:, None, :, :]
X_val   = X_val[:, None, :, :]
X_test  = X_test[:, None, :, :]

print("\nafter channel dim:")
print("X_train:", X_train.shape, X_train.dtype)
print("X_val:", X_val.shape, X_val.dtype)
print("X_test:", X_test.shape, X_test.dtype)

before channel dim:
X_train: (6139, 412, 424) float32
X_val: (1313, 412, 424) float32
X_test: (1332, 412, 424) float32

after channel dim:
X_train: (6139, 1, 412, 424) float32
X_val: (1313, 1, 412, 424) float32
X_test: (1332, 1, 412, 424) float32


## 8. Create simple pytorch dataset

In [75]:
class FullFieldAutoencoderDataset(Dataset):
    def __init__(self, data: np.ndarray):
        if data.ndim != 4:
            raise ValueError(f"Expected shape [N, C, H, W], got {data.shape}")
        self.data = torch.from_numpy(data)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx]
    
train_dataset = FullFieldAutoencoderDataset(X_train)
val_dataset   = FullFieldAutoencoderDataset(X_val)
test_dataset  = FullFieldAutoencoderDataset(X_test)